# Imports

In [11]:
try:
  import google.colab
  !pip install split-folders
  !pip install onnx
  !pip install onnxruntime
  !pip install onnxscript
except ImportError:
  pass

In [12]:
import torch as torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from tqdm.notebook import tqdm
import pandas as pd
import numpy as np
import cv2
import os
import time
import random
import gc
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report
from PIL import Image
from collections import OrderedDict
import shutil
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
from datetime import datetime
import scipy.stats as ss
import splitfolders
from torch.amp import GradScaler as GradScaler
import torch.onnx
import onnxruntime as ort
import onnxscript
import onnx


## utils.py

In [13]:
# All hyperparameters in one place

@dataclass
class Config:
    """All hyperparameters for Prototype 9."""
    # Data
    height: int = 128
    width: int = 128
    channels: int = 6
    fps: int = 10
    duration: int = 2
    seq_len: int = 20  # fps * duration

    # Training
    batch_size: int = 2
    accumulation_steps: int = 4  # Effective batch = 6 * 4 = 24
    num_epochs: int = 5
    learning_rate: float = 1e-4
    seed: int = 8

    # Early Stopping
    early_stop_patience: int = 5
    min_delta: float = 0.01  # 0.01% improvement threshold

    # Regularization
    dropout_rate: float = 0.5
    max_grad_norm: float = 1.0

    # LR Scheduler
    lr_factor: float = 0.5
    lr_patience: int = 3
    lr_min: float = 1e-7

    # Model Architecture
    input_dim: int = 6
    hidden_dim: List[int] = None  # [64, 32]
    kernel_size: Tuple[int, int] = (3, 3)
    num_layers: int = 2
    num_classes: int = 3

    # Cache
    reserve_gb: float = 10.0
    eviction_check_interval: int = 10
    eviction_buffer_percent: float = 0.10

    def __post_init__(self):
        if self.hidden_dim is None:
            self.hidden_dim = [64, 32]
        self.seq_len = self.fps * self.duration

# Global config instance
CONFIG = Config()

# Convenience aliases for backward compatibility
HEIGHT = CONFIG.height
WIDTH = CONFIG.width
CHANNELS = CONFIG.channels
FPS = CONFIG.fps
DURATION = CONFIG.duration
SEQ_LEN = CONFIG.seq_len

# MVO Prediction Logic Mapping
# FRONT: 0, LEFT: 1, RIGHT: 2
def get_label_id(label_name: str) -> int:
    """Map label name to numeric ID."""
    mapping = {'front': 0, 'left': 1, 'right': 2}
    return mapping.get(label_name.lower(), 0)

# Intent files
VAL_POSITIONS = ''
TEST_POSITIONS = ''

# Environment Detection
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Data Paths - Environment-specific configuration
if IN_COLAB:
    print("✓ Running in Google Colab")
    # Use if the dataset is in a zip file
    # Input the file name with the file extension (.zip)
    !unzip -q 'Dataset_750.zip'

    # Local SSD (fast, cleared on runtime disconnect)
    # DATA_DIR contains both and only video and label directories
    DATA_DIR = '/content/Dataset_750' # Folder names are strictly "videos" and "labels"
    CACHE_DIR = '/content/cache'
    VIDEO_DIR = os.path.join(DATA_DIR, "videos")
    LABEL_DIR = os.path.join(DATA_DIR, "labels")
else:
    print("✓ Running locally")
    # Local paths - adjust these to match your local setup
    DATA_DIR = r'C:\Users\User\Videos\aigd_10'
    CACHE_DIR = r'C:\Users\User\Videos\aigd_10_cache'
    VIDEO_DIR = os.path.join(DATA_DIR, "videos")
    LABEL_DIR = os.path.join(DATA_DIR, "labels")

# Validate paths exist
for path, name in [(VIDEO_DIR, "VIDEO_DIR"), (LABEL_DIR, "LABEL_DIR")]:
    if not os.path.exists(path):
        print(f"WARNING: {name} not found: {path}")

✓ Running in Google Colab


# cache_manager.py

In [14]:
class CacheManager:
    """
    Dynamic Cache Manager with LRU (Least Recently Used) Eviction.

    Manages a bounded cache directory, automatically evicting oldest-accessed
    files when the cache exceeds the configured size limit.

    Features:
    - Auto-detection: Automatically detects available storage and reserves space
    - Warm caching: Pre-caches videos up to the limit before training starts
    - On-demand caching: Videos are cached only when first accessed
    - LRU eviction: Oldest-accessed files deleted first when limit exceeded
    - O(1) operations: All cache operations are constant time
    """

    def __init__(
        self,
        cache_dir: str,
        max_size_gb: Optional[float] = None,
        transforms: Optional[Any] = None,
        num_frames: int = 20,
        eviction_check_interval: int = 10,
        eviction_buffer_percent: float = 0.10,
        auto_detect: bool = True,
        reserve_gb: float = 10.0
    ) -> None:
        """
        Initialize the CacheManager.

        Args:
            cache_dir: Directory to store cached .npy files
            max_size_gb: Maximum cache size in GB (None = auto-detect)
            transforms: Torchvision transforms to apply when caching
            num_frames: Number of frames per video
            eviction_check_interval: Check for eviction every N cache misses
            eviction_buffer_percent: Extra space to free during eviction (0.10 = 10%)
            auto_detect: If True, auto-detect available storage
            reserve_gb: GB to keep free when auto-detecting
        """
        self.cache_dir = cache_dir
        self.transforms = transforms
        self.num_frames = num_frames
        self.eviction_check_interval = eviction_check_interval
        self.eviction_buffer_percent = eviction_buffer_percent
        self.reserve_bytes = reserve_gb * (1024 ** 3)

        os.makedirs(cache_dir, exist_ok=True)

        # Auto-detect or use provided max_size_gb
        if auto_detect and max_size_gb is None:
            self.max_size_bytes = self._calculate_max_cache_size()
            self.auto_detected = True
        else:
            self.max_size_bytes = (max_size_gb or 7.5) * (1024 ** 3)
            self.auto_detected = False

        # Incremental size tracking - O(1)
        self.current_size_bytes: int = 0

        # LRU tracking with OrderedDict - O(1) for all operations
        self.lru_cache: OrderedDict[str, int] = OrderedDict()

        # Statistics
        self.hits: int = 0
        self.misses: int = 0
        self.evictions: int = 0
        self.warm_cached: int = 0
        self.misses_since_eviction_check: int = 0

        # Initialize from existing files
        self._initialize_from_disk()

        detect_str = "(auto-detected)" if self.auto_detected else "(configured)"
        print(f"✓ CacheManager initialized {detect_str}")
        print(f"  - Max size: {self.max_size_bytes / (1024**3):.1f} GB")
        print(f"  - Current: {self.get_cache_size_gb():.2f} GB ({len(self.lru_cache)} files)")

    def _calculate_max_cache_size(self) -> int:
        """Calculate maximum cache size based on available disk space."""
        try:
            disk_usage = shutil.disk_usage(self.cache_dir)
            max_cache_bytes = max(0, disk_usage.free - self.reserve_bytes)

            print(f"✓ Storage auto-detection:")
            print(f"  - Free: {disk_usage.free / (1024**3):.1f} GB")
            print(f"  - Reserved: {self.reserve_bytes / (1024**3):.1f} GB")
            print(f"  - Available for cache: {max_cache_bytes / (1024**3):.1f} GB")

            return max_cache_bytes
        except Exception as e:
            print(f"⚠ Auto-detect failed: {e}. Using 7.5 GB default.")
            return int(7.5 * (1024 ** 3))

    def _initialize_from_disk(self) -> None:
        """Load existing cache files and calculate initial size."""
        if not os.path.exists(self.cache_dir):
            return

        cached_files = []
        for filename in os.listdir(self.cache_dir):
            if filename.endswith('.npy'):
                filepath = os.path.join(self.cache_dir, filename)
                file_size = os.path.getsize(filepath)
                mtime = os.path.getmtime(filepath)
                cached_files.append((filename, file_size, mtime))
                self.current_size_bytes += file_size

        # Sort by modification time (oldest first)
        cached_files.sort(key=lambda x: x[2])
        for filename, file_size, _ in cached_files:
            self.lru_cache[filename] = file_size

    def get_cache_size_gb(self) -> float:
        """Get current cache size in GB."""
        return self.current_size_bytes / (1024 ** 3)

    def get_or_create(self, video_name: str, video_path: str) -> torch.Tensor:
        """
        Get cached tensor or create it from video.

        Args:
            video_name: Name of the video file (e.g., 'video_001.mp4')
            video_path: Full path to the video file

        Returns:
            Video frames tensor [num_frames, 3, H, W]
        """
        cache_filename = video_name.replace('.mp4', '.npy')
        cache_path = os.path.join(self.cache_dir, cache_filename)

        # CACHE HIT
        if cache_filename in self.lru_cache:
            self.hits += 1
            self.lru_cache.move_to_end(cache_filename)  # O(1)
            return torch.from_numpy(np.load(cache_path))

        # CACHE MISS
        self.misses += 1
        self.misses_since_eviction_check += 1
        video_tensor = self._decode_video(video_path)

        # Save to cache
        np.save(cache_path, video_tensor.numpy())
        file_size = os.path.getsize(cache_path)
        self.lru_cache[cache_filename] = file_size
        self.current_size_bytes += file_size

        # Lazy eviction
        if self.misses_since_eviction_check >= self.eviction_check_interval:
            self._batch_evict_if_needed()
            self.misses_since_eviction_check = 0

        return video_tensor

    def _decode_video(self, video_path: str) -> torch.Tensor:
        """Decode video file to tensor."""
        cap = cv2.VideoCapture(video_path)
        frames = []

        for _ in range(self.num_frames):
            ret, frame = cap.read()
            if not ret:
                frame_tensor = torch.zeros((3, HEIGHT, WIDTH))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if self.transforms:
                    frame = Image.fromarray(frame)
                    frame_tensor = self.transforms(frame)
                else:
                    frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
            frames.append(frame_tensor)

        cap.release()
        return torch.stack(frames, dim=0)

    def _batch_evict_if_needed(self) -> None:
        """Batch eviction with buffer."""
        if self.current_size_bytes <= self.max_size_bytes:
            return

        excess_bytes = self.current_size_bytes - self.max_size_bytes
        buffer_bytes = self.max_size_bytes * self.eviction_buffer_percent
        bytes_to_evict = excess_bytes + buffer_bytes

        bytes_evicted = 0
        files_to_remove = []

        for filename, file_size in self.lru_cache.items():
            if bytes_evicted >= bytes_to_evict:
                break
            files_to_remove.append((filename, file_size))
            bytes_evicted += file_size

        for filename, file_size in files_to_remove:
            filepath = os.path.join(self.cache_dir, filename)
            if os.path.exists(filepath):
                os.remove(filepath)
            del self.lru_cache[filename]
            self.current_size_bytes -= file_size
            self.evictions += 1

    def warm_cache(self, video_folder: str, max_videos: Optional[int] = None) -> int:
        """Pre-cache videos up to storage limit before training."""
        print(f"\n{'='*40}\nWARM CACHE\n{'='*40}")

        if self.auto_detected:
            self._calculate_max_cache_size()

        video_list = [f for f in os.listdir(video_folder) if f.endswith('.mp4')]
        if max_videos:
            video_list = video_list[:max_videos]

        uncached = [v for v in video_list if v.replace('.mp4', '.npy') not in self.lru_cache]
        print(f"To cache: {len(uncached)} videos")

        avg_size = sum(self.lru_cache.values()) / len(self.lru_cache) if self.lru_cache else SEQ_LEN * 3 * HEIGHT * WIDTH * 4
        cached_count = 0

        for video_name in tqdm(uncached, desc="Caching"):
            if (self.max_size_bytes - self.current_size_bytes) < avg_size * 1.1:
                print(f"⚠ Storage limit reached. Cached {cached_count} videos.")
                break

            video_path = os.path.join(video_folder, video_name)
            cache_filename = video_name.replace('.mp4', '.npy')
            cache_path = os.path.join(self.cache_dir, cache_filename)

            try:
                video_tensor = self._decode_video(video_path)
                np.save(cache_path, video_tensor.numpy())
                file_size = os.path.getsize(cache_path)
                self.lru_cache[cache_filename] = file_size
                self.current_size_bytes += file_size
                cached_count += 1
                self.warm_cached += 1
                avg_size = (avg_size * 0.9) + (file_size * 0.1)
                del video_tensor
            except Exception as e:
                print(f"⚠ Failed: {video_name}: {e}")

        print(f"✓ Cached {cached_count} videos ({self.get_cache_size_gb():.2f} GB)")
        return cached_count

    def get_stats(self) -> Dict[str, Any]:
        """Get cache statistics."""
        total = self.hits + self.misses
        return {
            'hits': self.hits,
            'misses': self.misses,
            'hit_rate': (self.hits / total * 100) if total > 0 else 0,
            'evictions': self.evictions,
            'cache_size_gb': self.get_cache_size_gb(),
            'num_files': len(self.lru_cache)
        }

    def reset_epoch_stats(self) -> None:
        """Reset hit/miss/eviction counters for new epoch."""
        self.hits = 0
        self.misses = 0
        self.evictions = 0
        self.misses_since_eviction_check = 0

    def print_stats(self, epoch: Optional[int] = None) -> None:
        """Print formatted cache statistics."""
        stats = self.get_stats()
        prefix = f"Epoch {epoch} " if epoch else ""
        print(f"{prefix}Cache: Hits={stats['hits']} Miss={stats['misses']} "
              f"Rate={stats['hit_rate']:.0f}% Evict={stats['evictions']}")


# Initialize CacheManager with config
cache_transforms = transforms.Compose([
    transforms.Resize((HEIGHT, WIDTH)),
    transforms.ToTensor()
])

cache_manager = CacheManager(
    cache_dir=CACHE_DIR,
    max_size_gb=None,
    transforms=cache_transforms,
    num_frames=SEQ_LEN,
    eviction_check_interval=CONFIG.eviction_check_interval,
    eviction_buffer_percent=CONFIG.eviction_buffer_percent,
    auto_detect=True,
    reserve_gb=CONFIG.reserve_gb
)

# Optional: Pre-cache videos
cache_manager.warm_cache(VIDEO_DIR)

✓ Storage auto-detection:
  - Free: 66.2 GB
  - Reserved: 10.0 GB
  - Available for cache: 56.2 GB
✓ CacheManager initialized (auto-detected)
  - Max size: 56.2 GB
  - Current: 2.75 GB (750 files)

WARM CACHE
✓ Storage auto-detection:
  - Free: 66.2 GB
  - Reserved: 10.0 GB
  - Available for cache: 56.2 GB
To cache: 0 videos


Caching: 0it [00:00, ?it/s]

✓ Cached 0 videos (2.75 GB)


0

## conv_lstm_classifier.py

In [15]:
# ConvLSTM Architecture for Video Direction Classification
#
# MODEL REQUIREMENTS:
# -------------------
# Input Shape:  [Batch, Seq_Len, Channels, Height, Width]
#               [B, 30, 6, 128, 128] for this config
#
# LAYER STRUCTURE (Config: num_layers=2, hidden_dim=[64, 32]):
# -------------------
# Layer 1: ConvLSTMCell
#   - Input channels:  3 (RGB frames) + 3 (intent)
#   - Hidden channels: 64
#   - Kernel size:     3×3
#   - Output:          [B, 30, 64, 128, 128]
#
# Layer 2: ConvLSTMCell
#   - Input channels:  64 (from Layer 1)
#   - Hidden channels: 32
#   - Kernel size:     3×3
#   - Output:          [B, 30, 32, 128, 128]
#
# Classification Head:
#   - Takes last time step: [B, 32, 128, 128]
#   - Flattens to:          [B, 524288]  (32 × 128 × 128)
#   - Dropout (0.5)
#   - Linear layer:         [524288 → 3]
#   - Output logits:        [B, 3] (Front, Left, Right)
#
# KERNEL OPERATIONS:
# -------------------
# Each ConvLSTMCell uses 3×3 convolutions with:
#   - Padding: 1 (preserves spatial dimensions)
#   - 4 parallel convolutions for LSTM gates: [input, forget, output, cell]
#   - Total parameters per cell: (input_dim + hidden_dim) × 4 × hidden_dim × 3 × 3
#
# Example Layer 1: (6 + 64) × 4 × 64 × 9 = 161,280 parameters
# Example Layer 2: (64 + 32) × 4 × 32 × 9 = 110,592 parameters

class ConvLSTMCell(nn.Module):
    """
    Single ConvLSTM memory unit - processes one frame at a time.

    Combines convolutional feature extraction with LSTM temporal memory.
    Uses 4 gates (input, forget, output, cell) to control information flow.

    Args:
        input_dim: Number of input channels (e.g., 6 for RGB and intent, 64 for hidden layer)
        hidden_dim: Number of hidden state channels (e.g., 64, 32)
        kernel_size: Size of convolutional kernel (e.g., (3, 3))
        bias: Whether to use bias in convolutions
    """

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        kernel_size: Tuple[int, int],
        bias: bool = True
    ) -> None:
        super(ConvLSTMCell, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.kernel_size = kernel_size
        self.padding = (kernel_size[0] // 2, kernel_size[1] // 2)
        self.bias = bias

        self.conv = nn.Conv2d(
            in_channels=self.input_dim + self.hidden_dim,
            out_channels=4 * self.hidden_dim,
            kernel_size=self.kernel_size,
            padding=self.padding,
            bias=self.bias
        )

    def forward(
        self,
        input_tensor: torch.Tensor,
        cur_state: Tuple[torch.Tensor, torch.Tensor]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Process one frame through ConvLSTM cell.

        LSTM Gate Equations:
        --------------------
        i_t = σ(W_xi * x_t + W_hi * h_{t-1})  ← Input gate (what to add)
        f_t = σ(W_xf * x_t + W_hf * h_{t-1})  ← Forget gate (what to keep)
        o_t = σ(W_xo * x_t + W_ho * h_{t-1})  ← Output gate (what to expose)
        g_t = tanh(W_xg * x_t + W_hg * h_{t-1}) ← Cell candidate

        c_t = f_t ⊙ c_{t-1} + i_t ⊙ g_t        ← Update cell state
        h_t = o_t ⊙ tanh(c_t)                   ← Update hidden state

        where ⊙ = element-wise multiplication, σ = sigmoid
        """
        h_cur, c_cur = cur_state  # Extract previous hidden and cell states

        # Concatenate input frame with previous hidden state
        # [B, input_dim, H, W] + [B, hidden_dim, H, W] → [B, input_dim+hidden_dim, H, W]
        combined = torch.cat([input_tensor, h_cur], dim=1)

        # Apply convolution to produce 4 gate activations
        # [B, input_dim+hidden_dim, H, W] → [B, 4×hidden_dim, H, W]
        combined_conv = self.conv(combined)

        # Split into 4 separate gates, each with hidden_dim channels
        # [B, 4×hidden_dim, H, W] → 4 tensors of [B, hidden_dim, H, W]
        cc_i, cc_f, cc_o, cc_g = torch.split(combined_conv, self.hidden_dim, dim=1)

        # Apply activation functions to each gate
        i = torch.sigmoid(cc_i)  # Input gate: how much new info to add (0-1)
        f = torch.sigmoid(cc_f)  # Forget gate: how much old info to keep (0-1)
        o = torch.sigmoid(cc_o)  # Output gate: how much to expose (0-1)
        g = torch.tanh(cc_g)     # Cell candidate: new information (-1 to 1)

        # Update cell state: forget old + add new
        c_next = f * c_cur + i * g

        # Update hidden state: apply output gate to cell state
        h_next = o * torch.tanh(c_next)

        return h_next, c_next

    def init_hidden(
        self,
        batch_size: int,
        image_size: Tuple[int, int]
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Initialize hidden and cell states with zeros.

        Returns:
            h_0: [batch_size, hidden_dim, height, width] - initial hidden state
            c_0: [batch_size, hidden_dim, height, width] - initial cell state

        Example for Layer 1: [B, 64, 128, 128]
        Example for Layer 2: [B, 32, 128, 128]
        """
        height, width = image_size
        device = self.conv.weight.device  # Match device of model parameters
        return (
            torch.zeros(batch_size, self.hidden_dim, height, width, device=device),
            torch.zeros(batch_size, self.hidden_dim, height, width, device=device)
        )


class ConvLSTM(nn.Module):
    """
    Multi-layer ConvLSTM - processes video sequences frame by frame.

    Stacks multiple ConvLSTMCell layers to form a deep temporal network.
    Each layer processes all frames sequentially, then passes to next layer.

    FORWARD PASS FLOW (num_layers=2, seq_len=30):
    ------------------------------------------------
    Input: [B, 30, 3, 128, 128]
      ↓
    Layer 1 processes frame-by-frame:
      ├─ Frame 1:  [B, 3, 128, 128] → h1, c1 [B, 64, 128, 128]
      ├─ Frame 2:  [B, 3, 128, 128] → h2, c2 [B, 64, 128, 128]
      ⋮
      └─ Frame 30: [B, 3, 128, 128] → h30, c30 [B, 64, 128, 128]
    Stack outputs → [B, 30, 64, 128, 128]
      ↓
    Layer 2 processes frame-by-frame:
      ├─ Frame 1:  [B, 64, 128, 128] → h1, c1 [B, 32, 128, 128]
      ├─ Frame 2:  [B, 64, 128, 128] → h2, c2 [B, 32, 128, 128]
      ⋮
      └─ Frame 30: [B, 64, 128, 128] → h30, c30 [B, 32, 128, 128]
    Stack outputs → [B, 30, 32, 128, 128]
      ↓
    Final output: Last layer's all time steps [B, 30, 32, 128, 128]

    Args:
        input_dim: Input channels (3 for RGB)
        hidden_dim: List of hidden dims per layer [64, 32]
        kernel_size: Kernel size for conv operations (3, 3)
        num_layers: Number of stacked ConvLSTM layers (2)
        batch_first: If True, input shape is [B, T, C, H, W]
        bias: Use bias in convolutions
        return_all_layers: If True, return outputs from all layers
    """

    def __init__(
        self,
        input_dim: int,               # 3 (RGB channels)
        hidden_dim: List[int],        # [64, 32] - one per layer
        kernel_size: Tuple[int, int], # (3, 3)
        num_layers: int,              # 2
        batch_first: bool = True,
        bias: bool = True,
        return_all_layers: bool = False
    ) -> None:
        super(ConvLSTM, self).__init__()

        # Validate and extend parameters for all layers
        self._check_kernel_size_consistency(kernel_size)
        kernel_size = self._extend_for_multilayer(kernel_size, num_layers)  # [(3,3), (3,3)]
        hidden_dim = self._extend_for_multilayer(hidden_dim, num_layers)    # [64, 32]

        self.input_dim = input_dim          # 3
        self.hidden_dim = hidden_dim        # [64, 32]
        self.kernel_size = kernel_size      # [(3,3), (3,3)]
        self.num_layers = num_layers        # 2
        self.batch_first = batch_first
        self.bias = bias
        self.return_all_layers = return_all_layers

        # Build list of ConvLSTM cells (one per layer)
        # Layer 0: input_dim=3, hidden_dim=64
        # Layer 1: input_dim=64, hidden_dim=32
        cell_list = []
        for i in range(self.num_layers):
            cur_input_dim = self.input_dim if i == 0 else self.hidden_dim[i - 1]
            cell_list.append(ConvLSTMCell(
                input_dim=cur_input_dim,
                hidden_dim=self.hidden_dim[i],
                kernel_size=self.kernel_size[i],
                bias=self.bias
            ))
        self.cell_list = nn.ModuleList(cell_list)

    def forward(
        self,
        input_tensor: torch.Tensor,  # [B, 30, 3, 128, 128]
        hidden_state: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None
    ) -> Tuple[List[torch.Tensor], List[Tuple[torch.Tensor, torch.Tensor]]]:
        """
        Forward pass through all ConvLSTM layers.

        Process:
        --------
        1. For each layer (2 layers):
           2. Initialize hidden states if not provided
           3. Process all 30 frames sequentially
           4. Stack frame outputs → [B, 30, hidden_dim, H, W]
           5. Use this as input to next layer

        Returns:
        --------
        layer_output_list: List of outputs from each layer
                          [[B, 30, 32, 128, 128]] if return_all_layers=False
                          [[B, 30, 64, 128, 128], [B, 30, 32, 128, 128]] if True

        last_state_list: List of final (h, c) for each layer
                        [(h_layer1, c_layer1), (h_layer2, c_layer2)]
        """
        # Convert to batch_first format if needed
        if not self.batch_first:
            # [T, B, C, H, W] → [B, T, C, H, W]
            input_tensor = input_tensor.permute(1, 0, 2, 3, 4)

        # Extract dimensions
        b, seq_len, _, h, w = input_tensor.size()  # [B=6, T=30, C=3, H=128, W=128]

        # Initialize hidden states for all layers if not provided
        if hidden_state is None:
            hidden_state = self._init_hidden(batch_size=b, image_size=(h, w))

        layer_output_list = []
        last_state_list = []
        cur_layer_input = input_tensor  # Start with raw video [B, 30, 3, 128, 128]

        # Process each layer sequentially
        for layer_idx in range(self.num_layers):
            h, c = hidden_state[layer_idx]  # Get initial states for this layer
            output_inner = []  # Store outputs for all time steps

            # Process each frame (time step) sequentially
            # This is where the RECURRENCE happens
            for t in range(seq_len):
                # Extract frame t: [B, C, H, W]
                input_t = cur_layer_input[:, t, :, :, :]

                # Process through ConvLSTM cell
                # Input: [B, C_in, H, W] + previous (h, c)
                # Output: new (h, c) with shape [B, hidden_dim, H, W]
                h, c = self.cell_list[layer_idx](input_tensor=input_t, cur_state=[h, c])

                # Save hidden state for this time step
                output_inner.append(h)

            # Stack all time steps: list of [B, hidden_dim, H, W] → [B, T, hidden_dim, H, W]
            layer_output = torch.stack(output_inner, dim=1)

            # This becomes input to next layer
            cur_layer_input = layer_output

            layer_output_list.append(layer_output)
            last_state_list.append([h, c])

        # If only returning final layer output (default)
        if not self.return_all_layers:
            layer_output_list = layer_output_list[-1:]  # Keep only last layer
            last_state_list = last_state_list[-1:]

        return layer_output_list, last_state_list

    def _init_hidden(
        self,
        batch_size: int,
        image_size: Tuple[int, int]
    ) -> List[Tuple[torch.Tensor, torch.Tensor]]:
        """Initialize hidden states for all layers."""
        return [self.cell_list[i].init_hidden(batch_size, image_size)
                for i in range(self.num_layers)]

    @staticmethod
    def _check_kernel_size_consistency(kernel_size: Any) -> None:
        """Validate kernel_size is tuple or list of tuples."""
        if not (isinstance(kernel_size, tuple) or
                (isinstance(kernel_size, list) and all(isinstance(elem, tuple) for elem in kernel_size))):
            raise ValueError('`kernel_size` must be tuple or list of tuples')

    @staticmethod
    def _extend_for_multilayer(param: Any, num_layers: int) -> List:
        """
        Extend parameter for all layers if single value provided.

        Example:
            (3, 3) with num_layers=2 → [(3, 3), (3, 3)]
            [64, 32] with num_layers=2 → [64, 32] (unchanged)
        """
        if not isinstance(param, list):
            param = [param] * num_layers
        return param


class ConvLSTMModel(nn.Module):
    """
    ConvLSTM classifier - processes video and outputs class prediction.

    FULL PIPELINE:
    --------------
    Input Video: [B, 30, 3, 128, 128]
        ↓
    ConvLSTM (2 layers):
        Layer 1: [B, 30, 3, 128, 128] → [B, 30, 64, 128, 128]
        Layer 2: [B, 30, 64, 128, 128] → [B, 30, 32, 128, 128]
        ↓
    Take Last Time Step: [B, 30, 32, 128, 128] → [B, 32, 128, 128]
        ↓
    Global Average Pooling: [B, 32, 128, 128] → [B, 32, 1, 1]
        ↓
    Flatten: [B, 32, 1, 1] → [B, 32]
        ↓
    Dropout(0.5): [B, 32] → [B, 32]
        ↓
    Linear: [B, 32] → [B, 3]
        ↓
    Output Logits: [B, 3] (Front=0, Left=1, Right=2)

    PARAMETERS (Optimized for Mobile):
    -----------
    ConvLSTM layers: ~265K parameters
    Linear layer:    32 × 3 = 96 parameters
    Total:           ~265K parameters (vs ~1.84M in original)

    MOBILE DEPLOYMENT BENEFITS:
    ---------------------------
    ✓ 80% smaller model size (better for app bundles)
    ✓ 10-20% faster inference (less computation)
    ✓ Lower memory footprint (better for edge devices)
    ⚠ Expected accuracy drop: 2-5% (acceptable trade-off)

    Args:
        input_dim: Input channels (3 for RGB)
        hidden_dim: Hidden dims per layer [64, 32]
        kernel_size: Conv kernel size (3, 3)
        num_layers: Number of ConvLSTM layers (2)
        height: Frame height (128)
        width: Frame width (128)
        num_classes: Output classes (3)
        dropout_rate: Dropout probability (0.5)
    """

    def __init__(
        self,
        input_dim: int,               # 3 (RGB)
        hidden_dim: List[int],        # [64, 32]
        kernel_size: Tuple[int, int], # (3, 3)
        num_layers: int,              # 2
        height: int,                  # 128
        width: int,                   # 128
        batch_first: bool = True,
        bias: bool = True,
        return_all_layers: bool = False,
        num_classes: int = 3,        # Front, Left, Right
        dropout_rate: float = 0.5
    ) -> None:
        super(ConvLSTMModel, self).__init__()

        # Build the ConvLSTM backbone
        self.convlstm = ConvLSTM(
            input_dim, hidden_dim, kernel_size, num_layers,
            batch_first=batch_first, bias=bias,
            return_all_layers=return_all_layers
        )

        # Global Average Pooling
        # Reduces spatial dimensions [B, C, H, W] → [B, C, 1, 1]
        # Eliminates 99.99% of FC layer parameters (1,572,867 → 99)
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Dropout for regularization (prevents overfitting)
        self.dropout = nn.Dropout(p=dropout_rate)

        # Classification head: maps pooled features to class logits
        # Input size: hidden_dim[-1] = 32 (after GAP reduces [32, 128, 128] to [32, 1, 1])
        # Output size: num_classes = 3
        # Parameters: 32 × 3 + 3 = 99 (vs 1,572,867 in original model)
        # Model size reduction: ~7.4 MB → ~1.5 MB (-80%)
        self.linear = nn.Linear(hidden_dim[-1], num_classes)

    def forward(
        self,
        input_tensor: torch.Tensor,  # [B, 30, 3, 128, 128]
        hidden_state: Optional[List[Tuple[torch.Tensor, torch.Tensor]]] = None
    ) -> torch.Tensor:
        """
        Full forward pass: video → ConvLSTM → classification.

        Steps:
        ------
        1. Process video through ConvLSTM layers
           [B, 30, 3, 128, 128] → [B, 30, 32, 128, 128]

        2. Take last time step (frame 30 contains all temporal info)
           [B, 30, 32, 128, 128] → [B, 32, 128, 128]

        3. Apply Global Average Pooling (Mobile Optimization)
           [B, 32, 128, 128] → [B, 32, 1, 1]
           Aggregates spatial information into channel-wise features

        4. Flatten pooled features
           [B, 32, 1, 1] → [B, 32]

        5. Apply dropout (training only)
           [B, 32] → [B, 32]

        6. Linear classification layer
           [B, 32] → [B, 3]

        Returns:
        --------
        logits: [B, 3] - raw scores for each class (before softmax)
        """
        # Process through ConvLSTM layers
        x, _ = self.convlstm(input_tensor)  # x is list, x[0] = [B, 30, 32, 128, 128]

        # Extract last time step (final frame has seen all previous frames)
        last_time_step = x[0][:, -1, :, :, :]  # [B, 32, 128, 128]

        # Apply Global Average Pooling to reduce spatial dimensions
        # [B, 32, 128, 128] → [B, 32, 1, 1]
        # This aggregates spatial information into channel-wise features
        pooled = self.global_avg_pool(last_time_step)

        # Flatten pooled features for classification
        # [B, 32, 1, 1] → [B, 32]
        flattened = torch.flatten(pooled, start_dim=1)

        # Apply dropout (active during training, disabled during eval)
        flattened = self.dropout(flattened)

        # Final linear layer: produce class logits
        return self.linear(flattened)  # [B, 3]

## dataset.py

In [16]:
class MVOVideoDataset(Dataset):
    """
    Video dataset with dynamic caching support.

    Loads 3-second video clips and their labels for turn prediction.
    Uses CacheManager for on-demand caching with LRU eviction.
    """

    def __init__(
        self,
        video_folder: str,
        label_folder: str,
        transforms: Optional[Any] = None,
        cache_manager: Optional[CacheManager] = None
    ) -> None:
        """
        Initialize the dataset.

        Args:
            video_folder: Path to video files
            label_folder: Path to label CSV files
            transforms: Torchvision transforms (used if no cache_manager)
            cache_manager: CacheManager instance for dynamic caching
        """
        self.video_folder = video_folder
        self.label_folder = label_folder
        self.transforms = transforms
        self.cache_manager = cache_manager

        # Find valid video-label pairs
        self.video_files: List[str] = []
        self.csv_files: List[str] = []
        try:
            all_videos = [f for f in os.listdir(video_folder) if f.endswith('.mp4')]
            for video_name in all_videos:
                csv_name = video_name.replace('.mp4', '.csv')
                csv_path = os.path.join(label_folder, csv_name)
                if os.path.exists(csv_path):
                    self.video_files.append(video_name)
                    self.csv_files.append(csv_name)
                else:
                    print(f"⚠ Missing label for {video_name}")

            cache_status = "✓ CacheManager enabled" if cache_manager else "⚠ No cache (slower)"
            print(f"{cache_status} | {len(self.video_files)} video-label pairs")

        except FileNotFoundError as e:
            print(f"Error: {e}")
            self.video_files = []
            self.csv_files = []

        self.split_type = ''
        self.positions = [] # If split_type == 'train', this would not be filled.

    def __len__(self) -> int:
        return len(self.video_files)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        video_name = self.video_files[idx]
        video_path = os.path.join(self.video_folder, video_name)

        # Load label
        csv_name = self.csv_files[idx]
        csv_path = os.path.join(self.label_folder, csv_name)
        df = pd.read_csv(csv_path)
        label = self._get_label(df)

        if self.split_type == 'TRAIN':
            intent_position = self._get_intent_position()
        else:
            intent_position = self.positions[idx]

        intent = self._get_intent(intent_position, df)

        # Load video (from cache or decode)
        if self.cache_manager is not None:
            video_tensor = self.cache_manager.get_or_create(video_name, video_path)

            # Split video tensor to frames
            frames = []
            for i in range(video_tensor.shape[0]):
                frame_tensor = video_tensor[i]
                frame_tensor = self._get_intent_torch(intent_position, i, intent, frame_tensor)
                frames.append(frame_tensor)

            video_tensor = torch.stack(frames, dim=0)
        else:
            video_tensor = self._load_from_video(video_name, intent, frame_tensor)

        return video_tensor, torch.tensor(label).long()

    def _load_from_video(self, video_name: str, intent_position: int, intent: int) -> torch.Tensor:
        """Load and process frames from video file (fallback when no cache)."""
        video_path = os.path.join(self.video_folder, video_name)
        cap = cv2.VideoCapture(video_path)
        frames = []

        for i in range(SEQ_LEN):
            ret, frame = cap.read()
            if not ret:
                frame_tensor = torch.zeros((3, HEIGHT, WIDTH))
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                if self.transforms:
                    frame = Image.fromarray(frame)
                    frame_tensor = self.transforms(frame)
                else:
                    frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0

            frame_tensor = self._get_intent_torch(intent_position, i, intent, frame_tensor)
            frames.append(frame_tensor)

        cap.release()
        return torch.stack(frames, dim=0)

    def _get_intent_torch(self, intent_position: int, i: int, intent: int, frame_tensor: torch.Tensor) -> torch.Tensor:
        # Create a tensor for the intent with the same spatial dimensions as the video frames
        # Used for no intent
        intent_torch = torch.zeros((3, HEIGHT, WIDTH))
        # If intent exists, add intent in its intent position for 1 second (10 frames)
        if intent_position != -1 and intent_position <= i and (intent_position + 10) > i:
            # Convert intent to one-hot vector by filling the specified channel with 1
            intent_torch[intent, :, :] = 1

        # Append the intent as a channel to the video frame
        frame_tensor = torch.cat((frame_tensor, intent_torch), dim=0)

        return frame_tensor

    def _get_label(self, df: pd.DataFrame) -> int:
        """Extract label from CSV using majority vote on last 24 frames."""
        col = 'label_id_corrected' if 'label_id_corrected' in df.columns else df.columns[-1]
        counts = df[col].tail(24).value_counts() # CSV files are in 24 fps

        label_counts = [counts.get(i, 0) for i in range(3)]

        if label_counts[0] == 24:
            return 0  # Front
        elif label_counts[1] > label_counts[2]:
            return 1  # Left
        elif label_counts[1] < label_counts[2]:
            return 2  # Right
        else:
            return int(df[col].tail(12).mode().iloc[0])

    def _get_intent_position(self) -> int:
        # 50% of the dataset have intent
        if random.random() < 0.5:
            # The time positions of the first 1 second (10 fps) because the intent is placed for a second
            start_frame = 0
            end_frame = 9
            median = (start_frame + end_frame)/2
            range_zero = np.arange(-median, median)

            # Obtain the probability of selecting a timestamp using the adjacent 0.5 areas
            smaller_range = range_zero - 0.5
            higher_range = range_zero + 0.5

            # Probability is the difference of the probability of higher range and lower range
            probability = ss.norm.cdf(higher_range) - ss.norm.cdf(smaller_range)

            # Normalize the probabilities
            # Each probability in probability range is divided by the sum of the probabilities in probability range
            probability /= probability.sum()

            # Select a timestamp based on the probabilities
            range = np.arange(start_frame, end_frame)
            intent_position = np.random.choice(range, p=probability)
        else:
            intent_position = -1

        return intent_position

    def _get_intent(self, intent_position: int, df: pd.DataFrame) -> int:
        # Check if the data has no intent
        if intent_position != -1:
            intent = self._get_label(df)
        else:
            intent = -1
        return intent

    def class_counter(self) -> Tuple[Dict[int, int], int]:
        """Count samples per class."""
        counts = {0: 0, 1: 0, 2: 0}
        for video_name in self.video_files:
            csv_path = os.path.join(self.label_folder, video_name.replace('.mp4', '.csv'))
            df = pd.read_csv(csv_path)
            counts[self._get_label(df)] += 1
        return counts, sum(counts.values())

    def set_split_type(self, dataset_type: str, len_dataset: int) -> None:
        global VAL_POSITIONS
        global TEST_POSITIONS
        self.split_type = dataset_type

        if self.split_type == 'VALIDATION':
            if VAL_POSITIONS == '':
                for _ in range(len_dataset):
                    self.positions.append(self._get_intent_position())
                np.save('val_intent_positions.npy', np.array(self.positions))
                VAL_POSITIONS = 'val_intent_positions.npy'
                print(f"Created {VAL_POSITIONS}")
            else:
                self.positions = list(np.load(VAL_POSITIONS))
                print(f"Loaded {VAL_POSITIONS}")
        elif self.split_type == 'TEST':
            if TEST_POSITIONS == '':
                for _ in range(len_dataset):
                    self.positions.append(self._get_intent_position())
                np.save('test_intent_positions.npy', np.array(self.positions))
                TEST_POSITIONS = 'test_intent_positions.npy'
                print(f"Created {TEST_POSITIONS}")
            else:
                self.positions = list(np.load(TEST_POSITIONS))
                print(f"Loaded {TEST_POSITIONS}")

        return None

## trainer.py

In [17]:
# Use centralized CONFIG
BATCH = CONFIG.batch_size
NUM_EPOCHS = CONFIG.num_epochs
LEARNING_RATE = CONFIG.learning_rate
ACCUMULATION_STEPS = CONFIG.accumulation_steps
EARLY_STOP_PATIENCE = CONFIG.early_stop_patience
MIN_DELTA = CONFIG.min_delta
SEED = CONFIG.seed
SAVED_MODEL_PATH = "best_convlstm.pth"
LOG_DIR = "runs"  # Directory for TensorBoard logs
CHECKPOINT_DIR = "checkpoint"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set seeds for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scaler: GradScaler,
    accumulation_steps: int = 1,
    max_grad_norm: float = 1.0
) -> Tuple[float, float, float]:
    """
    Train for one epoch with gradient accumulation and clipping.

    Returns:
        Tuple of (avg_loss, accuracy_pct, avg_gradient_norm)
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    total_grad_norm = 0.0
    num_updates = 0
    optimizer.zero_grad()

    loop = tqdm(loader, leave=True)
    for batch_idx, (data, targets) in enumerate(loop):
        data = data.float().to(DEVICE)
        targets = targets.to(DEVICE)

        # Forward
        if scaler is not None:
            with torch.autocast(device_type="cuda"):
                scores = model(data)
                loss = criterion(scores, targets) / accumulation_steps

            # Backward
            scaler.scale(loss).backward()
        else:
            # Forward
            scores = model(data)
            loss = criterion(scores, targets) / accumulation_steps

            # Backward
            loss.backward()

        # Update weights every accumulation_steps
        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            total_grad_norm += grad_norm.item()
            num_updates += 1
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()
            loop.set_description(f"Loss: {loss.item()*accumulation_steps:.4f}")

        # Stats
        running_loss += loss.item() * accumulation_steps
        _, predictions = scores.max(1)
        correct += (predictions == targets).sum().item()
        total += targets.size(0)

        del data, targets, scores, loss, predictions

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    avg_grad_norm = total_grad_norm / max(num_updates, 1)
    return running_loss / len(loader), 100 * correct / total, avg_grad_norm

def save_checkpoint(state: dict, dir: str) -> None:
   checkpoint_path = os.path.join(dir, 'checkpoint.pt')
   torch.save(state, checkpoint_path)

def load_checkpoint(checkpoint_path: str, model: nn.Module, optimizer: optim.Optimizer) -> tuple[nn.Module, optim.Optimizer, int]:
   checkpoint = torch.load(checkpoint_path)
   model.load_state_dict(checkpoint['state_dict'])
   optimizer.load_state_dict(checkpoint['optimizer'])
   return model, optimizer, checkpoint['epoch']

def main() -> None:
    """Main training loop."""
    # TensorBoard Setup - Create a unique run name with timestamp
    run_name = f"convlstm_proto7_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    writer = SummaryWriter(log_dir=os.path.join(LOG_DIR, run_name))

    # Print the actual directory where logs are being saved for easy access
    log_path = os.path.abspath(os.path.join(LOG_DIR, run_name))
    print(f"TensorBoard logging to: {log_path}")
    print(f"Run: py -m tensorboard.main --logdir=notebooks/runs or %tensorboard --logdir=runs")

    # Log hyperparameters as text
    hparams_text = f"""
    **Hyperparameters:**
    - Batch Size: {BATCH}
    - Accumulation Steps: {ACCUMULATION_STEPS} (Effective: {BATCH * ACCUMULATION_STEPS})
    - Learning Rate: {LEARNING_RATE}
    - Early Stop Patience: {EARLY_STOP_PATIENCE}
    - Min Delta: {MIN_DELTA}
    - Model Hidden Dims: {CONFIG.hidden_dim}
    - Kernel Size: {CONFIG.kernel_size}
    - Num Layers: {CONFIG.num_layers}
    - Dropout Rate: {CONFIG.dropout_rate}
    - Max Grad Norm: {CONFIG.max_grad_norm}
    """
    writer.add_text("Hyperparameters", hparams_text, 0)

    # Data setup
    transforms_train = transforms.Compose([
        transforms.Resize((HEIGHT, WIDTH)),
        transforms.ToTensor()
    ])

    """
    full_dataset = MVOVideoDataset(
        VIDEO_DIR, LABEL_DIR,
        transforms=transforms_train,
        cache_manager=cache_manager
    )

    # 60/20/20 split
    total_size = len(full_dataset)
    train_size = int(0.6 * total_size)
    val_size = int(0.2 * total_size)
    test_size = total_size - train_size - val_size

    generator = torch.Generator().manual_seed(SEED)
    train_dataset, val_dataset, _ = random_split(
        full_dataset, [train_size, val_size, test_size], generator=generator
    )
    """

    # Store paths of the split dataset
    dir = {'train': '', 'val': '', 'test': ''}
    dir_vid, dir_lbl = {}, {}
    for key in dir.keys():
        dir[key] = os.path.join("output", key)
        dir_vid[key] = os.path.join(dir[key], "videos")
        dir_lbl[key] = os.path.join(dir[key], "labels")

    # Split dataset if is not yet split
    if not (os.path.exists("output")):
        # Move the files from the input directory into three directories: training (80%), validation (20%), and testing (20%)
        print("Splitting files...")
        splitfolders.ratio(DATA_DIR, output="output", seed=8, ratio=(.6, .2, .2), group="sibling",move=True, shuffle=True)

    train_dataset = MVOVideoDataset(
        dir_vid['train'], dir_lbl['train'],
        transforms=transforms_train,
        cache_manager=cache_manager
    )

    val_dataset = MVOVideoDataset(
        dir_vid['val'], dir_lbl['val'],
        transforms=transforms_train,
        cache_manager=cache_manager
    )

    # Print distribution of dataset split
    print("-"*36)
    print(f"{"Split Dataset":^36}\n{"Train":^12}{"Validation":^12}{"Test":^12}")
    print(f"{len(train_dataset):^12}{len(val_dataset):^12}{len(os.listdir(dir_lbl['test'])):^12}")
    print("-"*36)

    train_dataset.set_split_type('TRAIN', len(train_dataset))
    val_dataset.set_split_type('VALIDATION', len(val_dataset))

    train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,
                             num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH, shuffle=False,
                           num_workers=0)

    # Calculate class instances for class weights
    label_counts, total_count = train_dataset.class_counter()
    class_weights = {'front': 0, 'left': 0, 'right': 0}

    # Calculate then print class weights
    print("-"*36)
    print(f"{"Training Dataset":^36}\n{"Class":^12}{"Instances":^12}{"Weights":^12}")
    for i, key in enumerate(class_weights.keys()):
        # Add 1 to avoid division by zero
        class_weights[key] = round(total_count / (label_counts[i] + 1), 2)
        print(f"{key.capitalize():^12}{label_counts[i]:^12}{class_weights[key]:^12}")
    print("-"*36)

    # Model
    model = ConvLSTMModel(
        input_dim=CONFIG.input_dim,
        hidden_dim=CONFIG.hidden_dim,
        kernel_size=CONFIG.kernel_size,
        num_layers=CONFIG.num_layers,
        height=HEIGHT,
        width=WIDTH,
        num_classes=CONFIG.num_classes,
        dropout_rate=CONFIG.dropout_rate
    ).to(DEVICE)

    # CrossEntropyLoss is used to handle class imbalance
    class_weights = torch.FloatTensor([class_weights['front'],class_weights['left'],class_weights['right']]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=CONFIG.lr_factor,
        patience=CONFIG.lr_patience, min_lr=CONFIG.lr_min
    )

    isMPT = False
    # Check if both GPU and autocast for the device are available
    if DEVICE == "cuda" and torch.amp.autocast_mode.is_autocast_available("cuda"):
        isMPT = True
        scaler = torch.amp.GradScaler("cuda")
        print("Training with automatic mixed precision.")

    # Loads a checkpoint
    checkpoint_path = os.path.join(CHECKPOINT_DIR, "checkpoint.pt")
    if os.path.exists(checkpoint_path):
        model, optimizer, start_epoch = load_checkpoint(checkpoint_path, model, optimizer)
        print("Successfully loaded a checkpoint.")
    else:
        start_epoch = 0

    # Training loop
    best_acc = 0.0
    epochs_no_improve = 0
    epoch_checkpoint = 2

    print(f"Training on {DEVICE}")
    print(f"Effective batch: {BATCH * ACCUMULATION_STEPS} | Early stop: {EARLY_STOP_PATIENCE} epochs")

    for epoch in range(NUM_EPOCHS):
        if start_epoch != 0:
            epoch = start_epoch + epoch

        cache_manager.reset_epoch_stats()
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS + start_epoch}")

        train_loss, train_acc, avg_grad_norm = train_one_epoch(
            model, train_loader, criterion, optimizer,
            scaler if isMPT else None,
            accumulation_steps=ACCUMULATION_STEPS,
            max_grad_norm=CONFIG.max_grad_norm
        )

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        val_loss = 0.0
        val_latencies = []

        all_preds: List[int] = []
        all_labels: List[int] = []

        with torch.no_grad():
            for batch_idx, (x, y) in enumerate(val_loader):
                x = x.float().to(DEVICE)
                y = y.to(DEVICE)

                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()
                start = time.perf_counter()

                scores = model(x)

                if DEVICE.type == 'cuda':
                    torch.cuda.synchronize()
                end = time.perf_counter()

                val_loss += criterion(scores, y).item()

                if batch_idx >= 3:  # Skip first 3 batches for warm-up
                    val_latencies.append(end - start)

                _, preds = scores.max(1)
                all_preds.extend(preds.detach().cpu().numpy())
                all_labels.extend(y.detach().cpu().numpy())

                val_correct += (preds == y).sum().item()
                val_total += y.size(0)

                del x, y, scores, preds

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        val_acc = 100 * val_correct / val_total
        val_loss_avg = val_loss / len(val_loader)
        avg_latency_ms = (np.mean(val_latencies) * 1000) if val_latencies else 0
        val_throughput = (1 / np.mean(val_latencies)) if val_latencies else 0
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Loss: {train_loss:.4f} | Train: {train_acc:.1f}% | Val: {val_acc:.1f}%")
        print(f"GradNorm: {avg_grad_norm:.3f} | Latency: {avg_latency_ms:.1f}ms | LR: {current_lr:.1e}")
        cache_manager.print_stats(epoch=epoch+1)
        print(classification_report(all_labels, all_preds, labels=[0, 1, 2], target_names=['Front', 'Left', 'Right'], zero_division=0))

        # TensorBoard Logging
        # Log Loss curves
        writer.add_scalars('Loss', {
            'Train': train_loss,
            'Validation': val_loss_avg
        }, epoch)

        # Log Accuracy curves
        writer.add_scalars('Accuracy', {
            'Train': train_acc,
            'Validation': val_acc
        }, epoch)

        # Log Learning Rate for monitoring scheduler behavior
        writer.add_scalar('Learning_Rate', current_lr, epoch)

        # Log Gradient Norm to monitor training stability
        writer.add_scalar('Gradient_Norm', avg_grad_norm, epoch)

        # Log Inference Performance
        writer.add_scalar('Inference/Latency_ms', avg_latency_ms, epoch)
        writer.add_scalar('Inference/Throughput_batches_per_sec', val_throughput, epoch)

        scheduler.step(val_acc)

        # Early stopping
        if val_acc > best_acc + MIN_DELTA:
            best_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), SAVED_MODEL_PATH)
            print(f"✓ Saved ({val_acc:.1f}%)")
        else:
            epochs_no_improve += 1
            print(f"No improvement ({epochs_no_improve}/{EARLY_STOP_PATIENCE})")
            if epochs_no_improve >= EARLY_STOP_PATIENCE:
                print(f"\n⚠ Early stop! Best: {best_acc:.1f}%")
                break

        # Saves a checkpoint
        if (epoch+1) % epoch_checkpoint == 0:
            checkpoint = {
                'epoch': epoch + 1,
                'state_dict': model.state_dict(),
                'optimizer': optimizer.state_dict()
            }

            try:
                os.mkdir(CHECKPOINT_DIR)
            except FileExistsError:
                pass

            save_checkpoint(checkpoint, CHECKPOINT_DIR)
            print(f"Saved a checkpoint at epoch {epoch+1} in '{CHECKPOINT_DIR}' directory.")

    # Log final best accuracy
    writer.add_scalar('Best_Validation_Accuracy', best_acc, NUM_EPOCHS)

    # Close TensorBoard writer
    writer.close()
    print(f"\n✓ TensorBoard logs saved to: {log_path}")
    print(f"\n✓ Done. Best accuracy: {best_acc:.1f}%")

if __name__ == "__main__":
    main()

TensorBoard logging to: /content/runs/convlstm_proto7_20260301_132748
Run: py -m tensorboard.main --logdir=notebooks/runs or %tensorboard --logdir=runs
✓ CacheManager enabled | 450 video-label pairs
✓ CacheManager enabled | 150 video-label pairs
------------------------------------
           Split Dataset            
   Train     Validation     Test    
    450         150         150     
------------------------------------
Created val_intent_positions.npy
------------------------------------
          Training Dataset          
   Class     Instances    Weights   
   Front        204         2.2     
    Left        133         3.36    
   Right        113         3.95    
------------------------------------
Successfully loaded a checkpoint.
Training on cuda
Effective batch: 8 | Early stop: 5 epochs

Epoch 5/9


  0%|          | 0/225 [00:00<?, ?it/s]

Loss: 0.6563 | Train: 68.7% | Val: 71.3%
GradNorm: 3.264 | Latency: 95.3ms | LR: 1.0e-04
Epoch 5 Cache: Hits=600 Miss=0 Rate=100% Evict=0
              precision    recall  f1-score   support

       Front       1.00      0.63      0.77        62
        Left       0.51      1.00      0.67        44
       Right       1.00      0.55      0.71        44

    accuracy                           0.71       150
   macro avg       0.84      0.72      0.72       150
weighted avg       0.86      0.71      0.72       150

✓ Saved (71.3%)

Epoch 6/9


  0%|          | 0/225 [00:00<?, ?it/s]

Loss: 0.5561 | Train: 70.2% | Val: 73.3%
GradNorm: 2.589 | Latency: 95.4ms | LR: 1.0e-04
Epoch 6 Cache: Hits=600 Miss=0 Rate=100% Evict=0
              precision    recall  f1-score   support

       Front       0.61      1.00      0.76        62
        Left       1.00      0.55      0.71        44
       Right       1.00      0.55      0.71        44

    accuracy                           0.73       150
   macro avg       0.87      0.70      0.72       150
weighted avg       0.84      0.73      0.73       150

✓ Saved (73.3%)
Saved a checkpoint at epoch 6 in 'checkpoint' directory.

Epoch 7/9


...

Loss: 0.6016 | Train: 68.9% | Val: 71.3%
GradNorm: 2.671 | Latency: 95.4ms | LR: 1.0e-04
Epoch 7 Cache: Hits=600 Miss=0 Rate=100% Evict=0
              precision    recall  f1-score   support

       Front       1.00      0.63      0.77        62
        Left       1.00      0.55      0.71        44
       Right       0.51      1.00      0.67        44

    accuracy                           0.71       150
   macro avg       0.84      0.72      0.72       150
weighted avg       0.86      0.71      0.72       150

No improvement (1/5)

Epoch 8/9


  0%|          | 0/225 [00:00<?, ?it/s]

Loss: 0.5395 | Train: 72.9% | Val: 73.3%
GradNorm: 2.339 | Latency: 95.3ms | LR: 1.0e-04
Epoch 8 Cache: Hits=600 Miss=0 Rate=100% Evict=0
              precision    recall  f1-score   support

       Front       0.61      1.00      0.76        62
        Left       1.00      0.55      0.71        44
       Right       1.00      0.55      0.71        44

    accuracy                           0.73       150
   macro avg       0.87      0.70      0.72       150
weighted avg       0.84      0.73      0.73       150

No improvement (2/5)
Saved a checkpoint at epoch 8 in 'checkpoint' directory.

Epoch 9/9


  0%|          | 0/225 [00:00<?, ?it/s]

Loss: 0.5399 | Train: 75.3% | Val: 73.3%
GradNorm: 3.196 | Latency: 95.3ms | LR: 1.0e-04
Epoch 9 Cache: Hits=600 Miss=0 Rate=100% Evict=0
              precision    recall  f1-score   support

       Front       0.61      1.00      0.76        62
        Left       1.00      0.55      0.71        44
       Right       1.00      0.55      0.71        44

    accuracy                           0.73       150
   macro avg       0.87      0.70      0.72       150
weighted avg       0.84      0.73      0.73       150

No improvement (3/5)

✓ TensorBoard logs saved to: /content/runs/convlstm_proto7_20260301_132748

✓ Done. Best accuracy: 73.3%


## tester.py

In [18]:
class Tester:
    """Evaluates trained model on test set with latency measurements."""

    def __init__(self, model_path: str, device: torch.device) -> None:
        self.model_path = model_path
        self.device = device
        self.transforms = transforms.Compose([
            transforms.Resize((HEIGHT, WIDTH)),
            transforms.ToTensor()
        ])

        self.model = ConvLSTMModel(
            input_dim=CONFIG.input_dim,
            hidden_dim=CONFIG.hidden_dim,
            kernel_size=CONFIG.kernel_size,
            num_layers=CONFIG.num_layers,
            height=HEIGHT,
            width=WIDTH,
            num_classes=CONFIG.num_classes,
            dropout_rate=CONFIG.dropout_rate
        ).to(self.device)

        self._load_weights()

    def _load_weights(self) -> None:
        """Load model weights and set to eval mode."""
        if os.path.exists(self.model_path):
            print(f"Loading {self.model_path}...")
            self.model.load_state_dict(torch.load(self.model_path, map_location=self.device))
            self.model.eval()
        else:
            raise FileNotFoundError(f"Model not found: {self.model_path}")

    def test(self) -> Tuple[List[int], List[int]]:
        """Run evaluation on test set."""
        print("Preparing test data...")

        """
        full_dataset = MVOVideoDataset(
            VIDEO_DIR, LABEL_DIR,
            transforms=self.transforms,
            cache_manager=cache_manager
        )

        # Recreate 60/20/20 split
        total_size = len(full_dataset)
        train_size = int(0.6 * total_size)
        val_size = int(0.2 * total_size)
        test_size = total_size - train_size - val_size

        generator = torch.Generator().manual_seed(CONFIG.seed)
        _, _, test_dataset = random_split(
            full_dataset, [train_size, val_size, test_size], generator=generator
        )
        """
        dir_test = os.path.join("output", "test")
        dir_test_vid = os.path.join(dir_test, "videos")
        dir_test_lbl = os.path.join(dir_test, "labels")

        test_dataset = MVOVideoDataset(
            dir_test_vid, dir_test_lbl,
            transforms=self.transforms,
            cache_manager=cache_manager
        )

        test_dataset.set_split_type('TEST', len(test_dataset))
        test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False,
                                num_workers=0)

        print(f"Evaluating {len(test_dataset)} videos...")

        all_preds: List[int] = []
        all_labels: List[int] = []
        latencies: List[float] = []

        with torch.no_grad():
            for i, (video, labels) in enumerate(tqdm(test_loader)):
                video = video.float().to(self.device)
                labels = labels.to(self.device)

                if self.device.type == 'cuda':
                    torch.cuda.synchronize()
                start = time.perf_counter()

                outputs = self.model(video)

                if self.device.type == 'cuda':
                    torch.cuda.synchronize()
                end = time.perf_counter()

                if i >= 5:  # Skip first 5 for warm-up
                    latencies.append(end - start)

                _, predicted = torch.max(outputs.data, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

                del video, labels, outputs, predicted

                if i % 50 == 0:
                    if self.device.type == 'cuda':
                        torch.cuda.empty_cache()
                    gc.collect()

        avg_latency_ms = np.mean(latencies) * 1000 if latencies else 0
        fps = 1 / np.mean(latencies) if latencies else 0

        self._print_metrics(all_labels, all_preds, avg_latency_ms, fps)
        self._save_results(all_labels, all_preds)

        return all_labels, all_preds

    def _print_metrics(
        self,
        y_true: List[int],
        y_pred: List[int],
        latency_ms: float,
        fps: float
    ) -> None:
        """Print evaluation metrics."""
        print(f"\n{'='*40}")
        print("PERFORMANCE REPORT")
        print(f"{'='*40}")
        print(f"Accuracy:  {accuracy_score(y_true, y_pred)*100:.2f}%")
        print(f"Precision: {precision_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
        print(f"Recall:    {recall_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")
        print(f"Latency:   {latency_ms:.2f} ms/clip | {fps:.1f} clips/sec")
        print(f"{'='*40}")
        print(classification_report(y_true, y_pred, labels=[0, 1, 2], target_names=['Front', 'Left', 'Right'], zero_division=0))

    def _save_results(self, y_true: List[int], y_pred: List[int]) -> None:
        """Save predictions to CSV."""
        label_map = {0: 'Front', 1: 'Left', 2: 'Right'}
        df = pd.DataFrame({
            'Actual': y_true,
            'Predicted': y_pred,
            'Actual_Text': [label_map[y] for y in y_true],
            'Predicted_Text': [label_map[y] for y in y_pred],
            'Correct': [a == p for a, p in zip(y_true, y_pred)]
        })
        df.to_csv("test_results.csv", index=False)
        print("✓ Results saved to test_results.csv")


if __name__ == "__main__":
    tester = Tester("best_convlstm.pth", DEVICE)
    tester.test()

Loading best_convlstm.pth...
Preparing test data...
✓ CacheManager enabled | 150 video-label pairs
Created test_intent_positions.npy
Evaluating 150 videos...


  0%|          | 0/150 [00:00<?, ?it/s]


PERFORMANCE REPORT
Accuracy:  71.33%
Precision: 0.8285
Recall:    0.7133
Latency:   47.20 ms/clip | 21.2 clips/sec
              precision    recall  f1-score   support

       Front       0.60      1.00      0.75        64
        Left       1.00      0.50      0.67        44
       Right       1.00      0.50      0.67        42

    accuracy                           0.71       150
   macro avg       0.87      0.67      0.69       150
weighted avg       0.83      0.71      0.70       150

✓ Results saved to test_results.csv


## Coversion to ONNX

In [ ]:
class DeployedModel:
    def __init__(self, model_path: str, onnx_path: str, device: torch.device) -> None:
        self.model_path = model_path
        self.onnx_path = onnx_path
        self.device = device

        self.model = ConvLSTMModel(
            input_dim=CONFIG.input_dim,
            hidden_dim=CONFIG.hidden_dim,
            kernel_size=CONFIG.kernel_size,
            num_layers=CONFIG.num_layers,
            height=HEIGHT,
            width=WIDTH,
            num_classes=CONFIG.num_classes,
            dropout_rate=CONFIG.dropout_rate
        ).to(self.device)

        self._load_weights()

    def _load_weights(self) -> None:
        """Load model weights and set to eval mode."""
        if os.path.exists(self.model_path):
            print(f"Loading {self.model_path}...")
            self.model.load_state_dict(torch.load(self.model_path, map_location=self.device))
            self.model.eval()
        else:
            raise FileNotFoundError(f"Model not found: {self.model_path}")

    def convert(self) -> None:
        # Convert from PyTorch to ONNX
        torch.onnx.export(
            self.model,
            self.get_input(),
            self.onnx_path,
            input_names=["video_input"],
            output_names=["direction_output"],
            # opset_version=12,
            # dynamic_axes={
            #    # Batch size is dynamic
            #    "video_input": {0: "batch_size"},
            #    "direction_output": {0: "batch_size"},
            #},
        )

        print(f"Success: conversion to ONNX. File is saved at: {self.onnx_path}")

    def inference(self) -> None:
        # Inference for the PyTorch model
        input = self.get_input()
        output = self.model(input)
        print("PyTorch Inference Output:", output.cpu().detach().numpy())

        # Load the ONNX model
        ort_session = ort.InferenceSession(self.onnx_path)

        # Inference for the ONNX model
        onnx_input = input.detach().cpu().numpy()
        onnx_output = ort_session.run(None, {"video_input": onnx_input})
        print(f"ONNX Inference Output: {onnx_output}")

    def get_input(self) -> torch.Tensor:
        """ Generate random input data with the same shape as the model's expected input """
        # Generate random video
        # Timesteps, Channels, Height, Width
        video_tensor = torch.rand(20, 3, HEIGHT, WIDTH).to(self.device)
        # Generate random intent
        intent_direction = random.randint(0, 2)
        # Generate random intent position
        intent_position = random.randint(0, 9)

        # Convert random video to frames
        frames = []
        for i in range(video_tensor.shape[0]):
            frame_tensor = video_tensor[i]

            intent_torch = torch.zeros((3, HEIGHT, WIDTH))
            # If intent exists, add intent in its intent position for 1 second (10 frames)
            if intent_position <= i and (intent_position + 10) > i:
                # Convert intent to one-hot vector by filling the specified channel with 1
                intent_torch[intent_direction, :, :] = 1

            intent_torch = intent_torch.to(frame_tensor.device)
            # Append the intent as a channel to the video frame
            frame_tensor = torch.cat((frame_tensor, intent_torch), dim=0)
            frames.append(frame_tensor)

        video_tensor = torch.stack(frames, dim=0)
        # Adds batch dimension to the tensor -> Batch, Timesteps, Channels, Height, Width
        final_video_tensor = video_tensor.unsqueeze(0)
        return final_video_tensor

    def speed(self) -> None:
        # PyTorch inference time (GPU)
        total_time = 0
        iterations = 100

        for _ in range(iterations):
            input = self.get_input()

            if self.device == "cuda":
                torch.cuda.synchronize()
            start_time = time.time()

            _ = self.model(input)

            if self.device == "cuda":
                torch.cuda.synchronize()
            end_time = time.time()
            total_time += (end_time - start_time)

        print("Speed of PyTorch and ONNX models in 1000 iterations:")
        print(f"PyTorch: {round(total_time/iterations, 2)} seconds")

        # ONNX inference time
        ort_session = ort.InferenceSession(self.onnx_path)
        total_time = 0
        iterations = 100

        output = None
        for _ in range(iterations):
            input = self.get_input().cpu().numpy().astype(np.float32)
            start_time = time.time()
            output = ort_session.run(None, {"video_input": input})
            end_time = time.time()
            total_time += (end_time - start_time)

        print(f"ONNX: {round(total_time/iterations, 2)} seconds")
        print(f"ONNX output shape: {output[0].shape}")

    def checker(self) -> None:
        onnx_model = onnx.load(self.onnx_path)
        try:
            onnx.checker.check_model(onnx_model)
            print("The model is successfully validated. ✅")
        except onnx.checker.ValidationError as e:
            print(f"An error occurred during model validation: {e}")

if __name__ == "__main__":
    model = DeployedModel("best_convlstm.pth", "convlstm.onnx", DEVICE)
    model.convert()
    model.inference()
    model.speed()
    model.checker()

/tmp/ipython-input-909/3152457656.py:31: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0301 13:39:15.247000 909 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


Loading best_convlstm.pth...


W0301 13:39:15.747000 909 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0301 13:39:15.749000 909 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0301 13:39:15.750000 909 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0301 13:39:15.756000 909 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `ConvLSTMModel([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ConvLSTMModel([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:65: adapt: Asserti

[torch.onnx] Translate the graph into ONNX... ✅
Applied 46 of general pattern rewrite rules.
Success: conversion to ONNX. File is saved at: convlstm.onnx
PyTorch Inference Output: [[-0.511606  -1.0289189  2.3158073]]
ONNX Inference Output: [array([[-0.511606 , -1.0289187,  2.315808 ]], dtype=float32)]
Speed of PyTorch and ONNX models in 1000 iterations:
PyTorch: 0.01 seconds
ONNX: 2.18 seconds
ONNX output shape: (1, 3)
The model is successfully validated. ✅
